# Дообучение Qwen-VL (OCR рунических надписей) — Kaggle

QLoRA / 4-bit. Обучение на синтетике (существующая + новые шарды с HF), оценка на gold set по CER.

**Перед запуском:**
1. Settings → **Internet = ON**, Accelerator → **GPU T4 ×2** (для 8B; для 2.5-7B хватает 1×T4).
2. Add-ons → **Secrets** → ключ `HF_TOKEN` (read; write нужен только если включишь выгрузку чекпойнтов).
3. Проверь `HF_SYNTH_REPO` и пути к датасетам в `CFG`.


In [ ]:
# === Зависимости (Qwen3-VL грузится через AutoModelForImageTextToText → transformers>=4.57) ===
!pip -q install -U "transformers>=4.57" "accelerate>=1.0" "peft>=0.13" \
                   "bitsandbytes>=0.46.1" qwen-vl-utils rapidfuzz

In [ ]:
# -*- coding: utf-8 -*-
import os, gc, re, glob, zipfile, random
from pathlib import Path
from dataclasses import dataclass
import numpy as np, pandas as pd, torch

# --- HF-токен из Kaggle Secrets (Add-ons → Secrets, ключ HF_TOKEN) ----------
HF_TOKEN='YOUR_HF_TOKEN'
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)

WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())


@dataclass
class CFG:
    # ── модель («Qwen3.5-VL-9B» не существует; взято ближайшее — Qwen3-VL-8B) ──
    MODEL_ID: str = "Qwen/Qwen3.5-9B"
    # ALT (лучший результат в ВКР, 54% CER):  "Qwen/Qwen2.5-VL-7B-Instruct"
    MAX_PIXELS: int = 384 * 384            # бюджет T4; для 2.5-7B можно 384–512

    # ── новые шарды синтетики на HF (3 потока → shard_0 / shard_1 / shard_2) ──
    HF_SYNTH_REPO: str = "AntoniusPerf/runic-synth-sd3-depth"   # ← проверь имя репо
    DOWNLOAD_FROM_HF: bool = True
    LOCAL_SHARD_DIRS: tuple = ("/kaggle/input",)   # если шарды добавлены через Add data

    # ── уже собранная синтетика (существующий Kaggle-датасет) ────────────────
    SYNTH_KAGGLE_DIR: str = "/kaggle/input/datasets/zhopa228/synth-final/synth_extracted_final"

    # ── реальный gold set для оценки ─────────────────────────────────────────
    GOLD_SOURCE: str = "/kaggle/input/datasets/zhopa228/val-dataset/val_dataset"
    GOLD_CSV_NAME: str = "real_corpus.csv"

    # ── обучение (QLoRA, проверенный конфиг) ─────────────────────────────────
    EPOCHS: int = 5
    LR: float = 1e-4
    BATCH: int = 1
    GRAD_ACCUM: int = 16
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05
    GEN_MAX_NEW_TOKENS: int = 64
    EARLY_STOPPING_PATIENCE: int = 3

    # ── сплиты / контроль утечки ─────────────────────────────────────────────
    VAL_FRAC: float = 0.05
    SEED: int = 1003
    EXCLUDE_GOLD_LEXICON: bool = True

    # ── метрики ──────────────────────────────────────────────────────────────
    STRIP_WHITESPACE_FOR_CER: bool = True
    STRIP_SEPARATORS_FOR_CER: bool = True
    BOOTSTRAP_B: int = 2000
    BOOTSTRAP_SEED: int = 0

    # ── продолжение обучения с чекпойнта + выгрузка чекпойнтов на HF ──────────
    RESUME_FROM: str = "/kaggle/input/datasets/zhopa228/ckpoint/checkpoint-123"          # путь к загруженному на Kaggle чекпойнту, напр.
                                   # "/kaggle/input/<ds>/checkpoint-123"; "" = с нуля
    PUSH_CKPT_TO_HF: bool = True   # каждый чекпойнт дублируется на HF Hub
    HF_CKPT_REPO: str = "AntoniusPerf/qwen-runic-checkpoints"   # ← repo для чекпойнтов

    OUT_DIR: str = "/kaggle/working/qwen_runic_run"


cfg = CFG()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(cfg.SEED)

In [ ]:
# === Разбор имён существующей синтетики + метрики (протокол как в ВКР) =======
_RUNIC_CHAR_RX = re.compile(r"[\u16A0-\u16FF]")

def _parse_synth_fname(name):
    """syn_w0_000006_translit_<руны>.png | syn_000006_translit_<руны>.png → dict|None."""
    stem = Path(name).stem
    m = _RUNIC_CHAR_RX.search(stem)
    if not m:
        return None
    pre = stem[: m.start()]
    if not pre.endswith("_"):
        return None
    runic = stem[m.start():]
    parts = pre[:-1].split("_", 3)
    if len(parts) < 3 or parts[0] != "syn":
        return None
    try:
        int(parts[1]); translit_raw = "_".join(parts[2:])      # без batch-ID
    except ValueError:
        if len(parts) < 4:
            return None
        try:
            int(parts[2])
        except ValueError:
            return None
        translit_raw = parts[3]                                # с batch-ID
    return {"translit": translit_raw.replace("_", " "), "runic": runic}


_WS_RX = re.compile(r"\s+")

def norm_cer(s):
    """Нормализация под CER: убрать пробелы (артефакт декодинга) и разделители слов."""
    s = (s or "").strip()
    s = _WS_RX.sub("", s) if cfg.STRIP_WHITESPACE_FOR_CER else _WS_RX.sub(" ", s)
    if cfg.STRIP_SEPARATORS_FOR_CER:
        for ch in "·:+":
            s = s.replace(ch, "")
    return s


def edit_distance(a, b):
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (0 if ai == b[j - 1] else 1))
        prev = cur
    return prev[lb]


def compute_cer(refs, preds):
    cd, cl = [], []
    for r, p in zip(refs, preds):
        r, p = norm_cer(r), norm_cer(p)
        cd.append(edit_distance(p, r)); cl.append(max(len(r), 1))
    cd, cl = np.asarray(cd), np.asarray(cl)
    return float(cd.sum() / max(cl.sum(), 1)), cd, cl


def bootstrap_cer(cd, cl, B=None, seed=None):
    B = B or cfg.BOOTSTRAP_B
    seed = cfg.BOOTSTRAP_SEED if seed is None else seed
    rng = np.random.default_rng(seed); n = len(cd)
    point = float(cd.sum() / max(cl.sum(), 1)); vals = np.empty(B)
    for k in range(B):
        idx = rng.integers(0, n, n)
        vals[k] = cd[idx].sum() / max(cl[idx].sum(), 1)
    return point, float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

In [ ]:
# === Сбор данных: новые шарды (HF) + существующая синтетика + gold set =======
def _collect_shard_zips():
    """Сначала ищем shard_*/batch_*.zip локально (Add data), иначе тянем с HF Hub."""
    zips = []
    for d in cfg.LOCAL_SHARD_DIRS:
        zips += glob.glob(f"{d}/**/shard_*/batch_*.zip", recursive=True)
    if not zips and cfg.DOWNLOAD_FROM_HF:
        from huggingface_hub import snapshot_download
        local = snapshot_download(repo_id=cfg.HF_SYNTH_REPO, repo_type="dataset",
                                  token=HF_TOKEN or None,
                                  allow_patterns=["shard_*/batch_*.zip"])
        zips = glob.glob(f"{local}/shard_*/batch_*.zip")
    print(f"шард-zip найдено: {len(zips)}")
    return sorted(zips)


def load_shards():
    """Распаковать images/ из каждого batch-zip, метки читать из labels_*.csv внутри zip."""
    base = WORK / "synth_shards"; img_dir = base / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for zp in _collect_shard_zips():
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            z.extractall(base, [n for n in names if n.startswith("images/")])
            csvs = [n for n in names if n.endswith(".csv")]
            if csvs:
                for c in csvs:
                    rows.append(pd.read_csv(z.open(c)))           # filename, runic, translit
            else:                                                  # fallback: метки из имён
                for n in names:
                    if n.startswith("images/") and n.endswith(".png"):
                        p = _parse_synth_fname(Path(n).name)
                        if p:
                            rows.append(pd.DataFrame([{"filename": Path(n).name, **p}]))
    if not rows:
        return pd.DataFrame(columns=["filename", "runic", "translit", "path"])
    df = pd.concat(rows, ignore_index=True)
    df["path"] = df["filename"].map(lambda fn: str(img_dir / fn))
    df = df[df["path"].map(lambda p: Path(p).exists())].reset_index(drop=True)
    print(f"новые шарды: {len(df)} строк")
    return df[["filename", "runic", "translit", "path"]]


def load_kaggle_synth():
    """Существующая синтетика: метки восстанавливаются из имён файлов."""
    root = Path(cfg.SYNTH_KAGGLE_DIR)
    if not root.exists():
        print("[i] существующая синтетика не найдена — пропуск")
        return pd.DataFrame()
    rows = []
    for p in root.rglob("*.png"):
        parsed = _parse_synth_fname(p.name)
        if parsed:
            rows.append({"filename": p.name, "path": str(p), **parsed})
    df = pd.DataFrame(rows)
    print(f"существующая синтетика: {len(df)} строк")
    return df


def load_gold_set():
    root = Path(cfg.GOLD_SOURCE)
    csvs = (glob.glob(f"{root}/**/{cfg.GOLD_CSV_NAME}", recursive=True)
            or glob.glob(f"{root}/**/*.csv", recursive=True))
    assert csvs, f"gold CSV не найден в {root}"
    df = pd.read_csv(sorted(csvs)[0])
    fcol = "filename" if "filename" in df.columns else "file_name"
    df = df.rename(columns={fcol: "file_name"})
    df["gt"] = df["translit"].astype(str)
    idx = {p.name: str(p) for p in root.rglob("*.png")}
    df["path"] = df["file_name"].map(idx.get)
    df = df[df["path"].notna()].reset_index(drop=True)
    print(f"gold set: {len(df)} строк")
    return df


def make_splits(synth, gold):
    """train/val с группировкой по уникальной транслитерации + исключение лексикона gold."""
    df = synth.copy(); df["translit"] = df["translit"].astype(str)
    if cfg.EXCLUDE_GOLD_LEXICON:
        gl = set(gold["gt"].map(lambda s: norm_cer(str(s))))
        n0 = len(df)
        df = df[~df["translit"].map(norm_cer).isin(gl)].reset_index(drop=True)
        print(f"исключено по лексикону gold set: {n0 - len(df)} строк")
    units = df["translit"].drop_duplicates().sample(frac=1.0, random_state=cfg.SEED).tolist()
    nv = max(1, int(len(units) * cfg.VAL_FRAC)); val_u = set(units[:nv])
    df["split"] = df["translit"].map(lambda t: "val" if t in val_u else "train")
    print(f"синтетика всего: {len(df)} | {df.split.value_counts().to_dict()}")
    return df


gold_df = load_gold_set()
synth_df = pd.concat([load_kaggle_synth(), load_shards()], ignore_index=True)
synth_df = synth_df.drop_duplicates("filename").reset_index(drop=True)
synth_df = make_splits(synth_df, gold_df)

In [ ]:
# === Qwen-VL: промпты, 4-бит, датасет, коллатор (маскирование промпта) =======
SYS_PROMPT = ("You are an expert runologist OCR system. You read runic inscriptions "
              "and output ONLY their transliteration in the Rundata convention, "
              "with no explanation.")
USR_PROMPT = "Transliterate the runic inscription in this image."


def _vlm_class():
    """Единый загрузчик для Qwen2-VL / Qwen2.5-VL / Qwen3-VL."""
    try:
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText
    except Exception:
        from transformers import AutoModelForVision2Seq
        return AutoModelForVision2Seq


def _qwen_bnb():
    from transformers import BitsAndBytesConfig
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=torch.float16,
                              llm_int8_skip_modules=["visual", "lm_head"])


def _qwen_msgs(image, answer=None):
    m = [{"role": "system", "content": SYS_PROMPT},
         {"role": "user", "content": [
             {"type": "image", "image": image, "max_pixels": cfg.MAX_PIXELS},
             {"type": "text", "text": USR_PROMPT}]}]
    if answer is not None:
        m.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return m


def _image_token_id(model, proc):
    tid = getattr(getattr(model, "config", None), "image_token_id", None)
    if tid is None:
        try:
            tid = proc.tokenizer.convert_tokens_to_ids("<|image_pad|>")
        except Exception:
            tid = -1
    return tid


class QwenDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return {"image": r["path"], "answer": str(r["translit"])}


def make_collator(proc, image_token_id):
    from qwen_vl_utils import process_vision_info

    def collate(batch):
        full = [_qwen_msgs(b["image"], b["answer"]) for b in batch]
        texts = [proc.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                 for m in full]
        imgs, _ = process_vision_info(full)
        enc = proc(text=texts, images=imgs, padding=True, return_tensors="pt")
        labels = enc["input_ids"].clone()
        labels[labels == proc.tokenizer.pad_token_id] = -100
        if image_token_id is not None and image_token_id >= 0:
            labels[labels == image_token_id] = -100
        # маскируем промпт — учим только ответ ассистента
        for i, b in enumerate(batch):
            pm = _qwen_msgs(b["image"], None)
            pt = proc.apply_chat_template(pm, tokenize=False, add_generation_prompt=True)
            pim, _ = process_vision_info(pm)
            plen = proc(text=[pt], images=pim, return_tensors="pt")["input_ids"].shape[1]
            labels[i, :plen] = -100
        enc["labels"] = labels
        return enc

    return collate

In [ ]:
# === Колбэк: каждый сохранённый чекпойнт → HF Hub ===========================
from transformers import TrainerCallback

class HFCheckpointCallback(TrainerCallback):
    """on_save: выгружает checkpoint-<step> из output_dir в cfg.HF_CKPT_REPO."""
    def on_save(self, args, state, control, **kwargs):
        if not (cfg.PUSH_CKPT_TO_HF and cfg.HF_CKPT_REPO):
            return control
        if not state.is_world_process_zero:
            return control
        from huggingface_hub import HfApi, upload_folder
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if not ckpt.exists():
            return control
        try:
            HfApi(token=HF_TOKEN or None).create_repo(
                cfg.HF_CKPT_REPO, repo_type="model", private=True, exist_ok=True)
            upload_folder(
                repo_id=cfg.HF_CKPT_REPO, repo_type="model",
                folder_path=str(ckpt), token=HF_TOKEN or None,
                path_in_repo=f"{Path(cfg.OUT_DIR).name}/checkpoint-{state.global_step}",
                commit_message=f"checkpoint step {state.global_step}")
            print(f"  ↑ HF: checkpoint-{state.global_step} → {cfg.HF_CKPT_REPO}")
        except Exception as e:
            print(f"  [!] выгрузка чекпойнта не удалась: {e}")
        return control

In [ ]:
# === Обучение QLoRA ==========================================================
def train_qwen(synth):
    from transformers import (AutoProcessor, Trainer, TrainingArguments,
                              EarlyStoppingCallback)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    set_seed(cfg.SEED)

    proc = AutoProcessor.from_pretrained(cfg.MODEL_ID, max_pixels=cfg.MAX_PIXELS)
    proc.tokenizer.padding_side = "right"

    model = _vlm_class().from_pretrained(
        cfg.MODEL_ID, quantization_config=_qwen_bnb(),
        torch_dtype=torch.float16, device_map="auto")
    # model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False

    model = get_peft_model(model, LoraConfig(
        r=cfg.LORA_R, lora_alpha=cfg.LORA_ALPHA, lora_dropout=cfg.LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    model.is_parallelizable = False             # device_map="auto" сам шардит на 2×T4

    collate = make_collator(proc, _image_token_id(model, proc))
    cbs = ([EarlyStoppingCallback(early_stopping_patience=cfg.EARLY_STOPPING_PATIENCE)]
           if cfg.EARLY_STOPPING_PATIENCE > 0 else [])
    cbs.append(HFCheckpointCallback())      # все чекпойнты → HF Hub

    args = TrainingArguments(
        output_dir=cfg.OUT_DIR, per_device_train_batch_size=cfg.BATCH,
        per_device_eval_batch_size=cfg.BATCH, gradient_accumulation_steps=cfg.GRAD_ACCUM,
        num_train_epochs=cfg.EPOCHS, learning_rate=cfg.LR, warmup_ratio=0.05, fp16=True,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, save_total_limit=2,
        logging_steps=20, dataloader_num_workers=2, remove_unused_columns=False,
        report_to="none", optim="paged_adamw_8bit")

    trainer = Trainer(model=model, args=args, data_collator=collate,
                      train_dataset=QwenDataset(synth[synth.split == "train"]),
                      eval_dataset=QwenDataset(synth[synth.split == "val"]),
                      callbacks=cbs)
    trainer.train(resume_from_checkpoint=(cfg.RESUME_FROM or None))

    adapter = str(Path(cfg.OUT_DIR) / "lora-adapter")
    model.save_pretrained(adapter); proc.save_pretrained(adapter)
    print("адаптер сохранён →", adapter)
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return adapter

In [ ]:
adapter_dir = train_qwen(synth_df)

In [ ]:
# === (опционально) Оценка на gold set: CER + бутстрэп-ДИ =====================
@torch.no_grad()
def infer_qwen(adapter_dir, paths):
    from transformers import AutoProcessor
    from peft import PeftModel
    from qwen_vl_utils import process_vision_info
    proc = AutoProcessor.from_pretrained(adapter_dir, max_pixels=cfg.MAX_PIXELS)
    base = _vlm_class().from_pretrained(
        cfg.MODEL_ID, quantization_config=_qwen_bnb(),
        torch_dtype=torch.float16, device_map="auto")
    model = (PeftModel.from_pretrained(base, adapter_dir).eval()
             if Path(adapter_dir, "adapter_config.json").exists() else base.eval())
    preds = []
    for p in paths:
        msgs = _qwen_msgs(str(p), None)
        text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        imgs, _ = process_vision_info(msgs)
        inp = proc(text=[text], images=imgs, return_tensors="pt").to(model.device)
        out = model.generate(**inp, max_new_tokens=cfg.GEN_MAX_NEW_TOKENS, do_sample=False)
        preds.append(proc.batch_decode(out[:, inp["input_ids"].shape[1]:],
                                       skip_special_tokens=True)[0].strip())
    del model, base; gc.collect(); torch.cuda.empty_cache()
    return preds


gold_preds = infer_qwen(adapter_dir, gold_df["path"].tolist())
cer, cd, cl = compute_cer(gold_df["gt"].astype(str).tolist(), gold_preds)
point, lo, hi = bootstrap_cer(cd, cl)
print(f"\nCER gold = {100 * point:.2f}%  (95% ДИ {100 * lo:.1f}–{100 * hi:.1f})  n={len(gold_df)}")

res = pd.DataFrame({"file": gold_df["file_name"], "gt": gold_df["gt"], "pred": gold_preds})
res["ok"] = res.apply(lambda r: "✓" if norm_cer(r["gt"]) == norm_cer(r["pred"]) else "", axis=1)
res.to_csv(Path(cfg.OUT_DIR) / "gold_predictions.csv", index=False, encoding="utf-8")
with pd.option_context("display.max_colwidth", 60, "display.max_rows", 200):
    print(res.to_string(index=True))